# 02 — Integração dos dados de qualidade do ar e meteorologia

Este notebook lê os JSONs brutos produzidos pela coleta, transforma as séries horárias em tabelas e integra as fontes pela coluna `time`. O resultado é salvo em `data/interim/` como CSV.

## O que será feito

1. Ler do YAML o período e os caminhos de entrada/saída.
2. Conferir se os dois JSONs existem e impedir que o CSV seja sobrescrito.
3. Abrir e inspecionar as tabelas horárias de cada fonte.
4. Comparar horários, verificar duplicatas e valores ausentes.
5. Fazer o merge interno (`inner`) pelo horário e salvar o resultado.

> O merge interno mantém apenas horários presentes nas duas fontes. Se a auditoria apontar horários sem correspondência, revise a coleta antes de prosseguir: o notebook informa quantos registros não entraram no resultado. Nenhum valor é imputado ou removido deliberadamente.

Execute as células em ordem. É necessário ter concluído [`01_Coleta_Dados.ipynb`](01_Coleta_Dados.ipynb) ou produzido os mesmos JSONs pelo script `src/coleta/Coleta_Dados.py`.

## 1. Carregar a configuração e determinar os arquivos

Os nomes dependem das datas configuradas, permitindo manter lado a lado coletas de períodos diferentes.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd()
if not (ROOT / "config" / "params.yaml").is_file():
    ROOT = ROOT.parent
CONFIG_PATH = ROOT / "config" / "params.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Não encontrei a configuração do projeto: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as arquivo:
    config = yaml.safe_load(arquivo)

collection = config["collection"]
periodo = {
    "start_date": collection["start_date"],
    "end_date": collection["end_date"],
}
paths = config["paths"]
air_quality_path = ROOT / paths["raw_data"] / paths["air_quality_raw_filename"].format(**periodo)
weather_path = ROOT / paths["raw_data"] / paths["weather_raw_filename"].format(**periodo)
merged_path = ROOT / paths["interim_data"] / paths["merged_filename"].format(**periodo)

print(f"Período configurado: {periodo['start_date']} a {periodo['end_date']}")
print(f"Entrada qualidade do ar: {air_quality_path}")
print(f"Entrada meteorologia:    {weather_path}")
print(f"Saída prevista:           {merged_path}")

## 2. Conferir entradas e evitar sobrescrita

Se os arquivos brutos estiverem ausentes, execute primeiro o notebook de coleta. Se a saída já existir, o processamento para para proteger o resultado anterior.

In [ ]:
entradas_ausentes = [str(path) for path in (air_quality_path, weather_path) if not path.is_file()]
if entradas_ausentes:
    raise FileNotFoundError("Arquivo(s) bruto(s) não encontrado(s): " + ", ".join(entradas_ausentes))
if merged_path.exists():
    raise FileExistsError(f"O merge já existe e não será sobrescrito: {merged_path}")

print("Os dois arquivos de entrada foram encontrados e o caminho de saída está livre.")

## 3. Ler os JSONs e montar as tabelas horárias

A seção `hourly` de cada resposta contém uma série de horários e uma coluna para cada variável solicitada. As verificações abaixo detectam estrutura inválida, séries de comprimentos diferentes e timestamps repetidos antes do merge.

In [ ]:
with air_quality_path.open("r", encoding="utf-8") as arquivo:
    dados_air_quality = json.load(arquivo)
with weather_path.open("r", encoding="utf-8") as arquivo:
    dados_weather = json.load(arquivo)

def criar_tabela_horaria(nome, resposta):
    hourly = resposta.get("hourly")
    if not isinstance(hourly, dict) or "time" not in hourly:
        raise ValueError(f"O JSON de {nome} não contém uma seção 'hourly.time' válida.")
    comprimentos = {coluna: len(valores) for coluna, valores in hourly.items()}
    if len(set(comprimentos.values())) != 1:
        raise ValueError(f"Séries com comprimentos diferentes em {nome}: {comprimentos}")

    tabela = pd.DataFrame(hourly)
    tabela["time"] = pd.to_datetime(tabela["time"], errors="raise")
    if tabela["time"].isna().any():
        raise ValueError(f"Há timestamps ausentes em {nome}.")
    if tabela["time"].duplicated().any():
        raise ValueError(f"Há timestamps duplicados em {nome}; não é seguro fazer o merge.")
    return tabela

df_air_quality = criar_tabela_horaria("qualidade do ar", dados_air_quality)
df_weather = criar_tabela_horaria("meteorologia", dados_weather)

print(f"Qualidade do ar: {df_air_quality.shape[0]:,} linhas × {df_air_quality.shape[1]} colunas")
display(df_air_quality.head())
print(f"Meteorologia: {df_weather.shape[0]:,} linhas × {df_weather.shape[1]} colunas")
display(df_weather.head())

## 4. Auditar a correspondência dos horários

Antes de integrar, comparamos quantos timestamps são comuns às duas fontes. Uma diferença não é preenchida automaticamente: o `inner join` abaixo descartaria horários sem par, e a contagem torna esse efeito explícito.

In [ ]:
horarios_air_quality = set(df_air_quality["time"])
horarios_weather = set(df_weather["time"])
horarios_comuns = horarios_air_quality & horarios_weather
somente_air_quality = horarios_air_quality - horarios_weather
somente_weather = horarios_weather - horarios_air_quality

print(f"Horários de qualidade do ar: {len(horarios_air_quality):,}")
print(f"Horários meteorológicos:    {len(horarios_weather):,}")
print(f"Horários em comum:          {len(horarios_comuns):,}")
print(f"Somente qualidade do ar:    {len(somente_air_quality):,}")
print(f"Somente meteorologia:       {len(somente_weather):,}")

if somente_air_quality or somente_weather:
    print("ATENÇÃO: o merge interno manterá somente os horários em comum. Investigue a diferença antes de continuar.")
else:
    print("As fontes possuem os mesmos timestamps.")

## 5. Integrar e conferir o resultado

O merge é feito pela chave temporal, exigindo relação um-para-um. Em seguida, são apresentados o tamanho, valores ausentes e duplicatas do resultado. Não há imputação nem eliminação automática de linhas.

In [ ]:
df_merged = pd.merge(
    df_air_quality,
    df_weather,
    on="time",
    how="inner",
    validate="one_to_one",
).sort_values("time").reset_index(drop=True)

nulos = df_merged.isna().sum()
duplicatas = int(df_merged["time"].duplicated().sum())
print(f"Resultado integrado: {df_merged.shape[0]:,} linhas × {df_merged.shape[1]} colunas")
print(f"Duplicatas de horário: {duplicatas}")
print("Colunas com valores ausentes:")
display(nulos[nulos > 0].rename("valores_ausentes").to_frame())
if nulos.sum() == 0:
    print("Nenhum valor ausente encontrado.")
display(df_merged.head())

## 6. Salvar em `data/interim/`

O CSV contém as variáveis horárias das duas fontes e a coluna `time`. O alvo CETESB e o rótulo de previsão são calculados em uma etapa separada.

In [ ]:
if merged_path.exists():
    raise FileExistsError(f"O merge já existe e não será sobrescrito: {merged_path}")
merged_path.parent.mkdir(parents=True, exist_ok=True)
df_merged.to_csv(merged_path, index=False, encoding="utf-8")

print(f"Merge salvo em: {merged_path}")
print(f"Registros de qualidade do ar: {len(df_air_quality):,}")
print(f"Registros meteorológicos:    {len(df_weather):,}")
print(f"Registros integrados:        {len(df_merged):,}")
print("A próxima etapa é calcular os subíndices CETESB e o alvo de t+1h.")

## Próxima etapa

O script [`Calcular_Alvo_IQAr.py`](../src/transformacao/Calcular_Alvo_IQAr.py) calcula os subíndices e o rótulo futuro usando o CSV integrado. A alternativa de terminal para repetir o merge está em `src/transformacao/Merge_Dados.py`.